# Exploration: XGBoost Features with Simple CNN

Plan for late fusion hybrid:

- **Branch 1: Engineered XGB Features**
- **Branch 2: CNN**
- **Concatenate**

[ CNN vector | tabular vector ] → Dense → prediction

**Input** = Tabular features + MelSpec images

**Output** = Category prediction (multiclass) with confidence rating

Notes: 

- Each sample in XGB feature set is a 10 Hz, 6s chunk
- Split by patient to avoid data leakage (test/train and validation) - patient ID in GroupKFold and GroupShuffleSplit
- Using **5 classes**: COPD, pneumonia, healthy, URTI, bronchiectasis
- CNN weighting TBC


### 📈 Running Summary

#### Tabular XGBoost baseline

Cross-validation summary:

| Fold | Accuracy | Macro F1 | Weighted F1 |
|------|----------|----------|-------------|
| 1    | 0.8125   | 0.4295   | 0.8342      |
| 2    | 0.8732   | 0.3911   | 0.8759      |
| 3    | 0.8852   | 0.4579   | 0.8891      |

Mean metrics:

- accuracy:       0.856986

- macro_f1:      0.426176

- weighted_f1:    0.866393

#### Simple CNN baseline

#### Hybrid v1

### Imports

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

## Branch 1: **Engineered XGBoost Features**

### Load XGB data

In [2]:
xgb_df = pd.read_csv("/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Extracted features/xgboost_features_6s_10kHz_compressed.csv")

In [3]:
xgb_df.head()

,rms_mean,zcr_mean,centroid_mean,rolloff_mean,flatness_mean,flux_mean,mfcc_1,mfcc_2,mfcc_3,mfcc_4,...,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,patient_id,chunk_id,original_file,diagnosis
0,0.739170,0.001539,101.449467,127.325543,0.000056,1.957479,141.853965,44.231221,31.555366,29.084454,...,15.377089,13.318531,9.304475,7.877657,7.429528,8.447693,223,0,223_1b1_Pr_sc_Meditron.wav,COPD
1,0.696589,0.001879,98.285502,113.049523,0.000059,1.518246,143.352698,39.783529,33.081844,32.168925,...,15.717680,15.663969,10.037776,7.375149,6.357885,8.668443,223,1,223_1b1_Pr_sc_Meditron.wav,COPD
2,0.670110,0.001899,94.122156,103.863215,0.000033,1.650601,143.955065,47.641450,33.539781,33.385107,...,14.671383,14.495760,10.442390,6.406775,5.649503,9.201296,223,2,223_1b1_Pr_sc_Meditron.wav,COPD
3,0.674821,0.001382,84.610496,86.525093,0.000015,1.759859,140.524004,48.073060,33.807260,31.479628,...,16.430892,14.911375,10.164741,6.695922,5.836122,8.794603,223,3,223_1b1_Pr_sc_Meditron.wav,COPD
4,0.634854,0.001432,85.512759,88.056144,0.000027,1.697848,133.707197,48.023790,28.651836,32.951737,...,18.016766,16.560348,10.037829,6.151356,4.874320,9.356361,223,4,223_1b1_Pr_sc_Meditron.wav,COPD


In [4]:
xgb_df.shape

(2978, 22)

In [5]:
xgb_df.columns

Index(['rms_mean', 'zcr_mean', 'centroid_mean', 'rolloff_mean',
       'flatness_mean', 'flux_mean', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4',
       'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11',
       'mfcc_12', 'patient_id', 'chunk_id', 'original_file', 'diagnosis'],
      dtype='object')

In [6]:
xgb_df.dtypes

rms_mean         float64
zcr_mean         float64
centroid_mean    float64
rolloff_mean     float64
flatness_mean    float64
flux_mean        float64
mfcc_1           float64
mfcc_2           float64
mfcc_3           float64
mfcc_4           float64
mfcc_5           float64
mfcc_6           float64
mfcc_7           float64
mfcc_8           float64
mfcc_9           float64
mfcc_10          float64
mfcc_11          float64
mfcc_12          float64
patient_id         int64
chunk_id           int64
original_file     object
diagnosis         object
dtype: object

In [7]:
xgb_df["diagnosis"].value_counts()

diagnosis
COPD              2597
Pneumonia          111
Healthy            105
URTI                69
Bronchiectasis      48
Bronchiolitis       39
LRTI                 6
Asthma               3
Name: count, dtype: int64

### XGB Preprocessing

(to align with Antonella's notebook)

In [8]:
# Filter to same 5 classes - COPD, pneumonia, healthy, URTI, bronchiectasis
classes_to_keep = ["COPD", "Pneumonia", "Healthy", "URTI", "Bronchiectasis"]

xgb_df_filtered = xgb_df[xgb_df["diagnosis"].isin(classes_to_keep)].copy()
xgb_df_filtered = xgb_df_filtered.reset_index(drop=True)

In [9]:
xgb_df_filtered.shape

(2930, 22)

In [10]:
xgb_df_filtered["diagnosis"].value_counts()

diagnosis
COPD              2597
Pneumonia          111
Healthy            105
URTI                69
Bronchiectasis      48
Name: count, dtype: int64

In [ ]:
# Encode target
le = LabelEncoder()
xgb_df_filtered["target"] = le.fit_transform(xgb_df_filtered["diagnosis"])

# Create a dictionary mapping labels -> numbers
class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

class_mapping

{'Bronchiectasis': 0, 'COPD': 1, 'Healthy': 2, 'Pneumonia': 3, 'URTI': 4}

### XGBoost baseline

In [14]:
# Define features and target
X = xgb_df_filtered.drop(columns=["original_file", "patient_id", "diagnosis", "target", "chunk_id"])
y = xgb_df_filtered["target"]

# Define groups (by patient)
groups = xgb_df_filtered["patient_id"]

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of groups (patients): {groups.nunique()}")
print("\nConfirm feature columns:")
print(list(X.columns))


Feature matrix shape: (2930, 18)
Target shape: (2930,)
Number of groups (patients): 117

Confirm feature columns:
['rms_mean', 'zcr_mean', 'centroid_mean', 'rolloff_mean', 'flatness_mean', 'flux_mean', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12']


In [ ]:
# Grouped Cross Validation (split by patient ID)
# Each fold trains on 2/3 of patients, tests on remaining 1/3
gkf = GroupKFold(n_splits=3)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    # Outer split: creates grouped train/test splits
    X_train_full = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train_full = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train_full = groups.iloc[train_idx]

    # Inner split: creates grouped train/validation splits
    gss_val = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
    idx_train, idx_val = next(gss_val.split(X_train_full, y_train_full, groups=groups_train_full))

    X_train = X_train_full.iloc[idx_train]
    X_val = X_train_full.iloc[idx_val]

    y_train = y_train_full.iloc[idx_train]
    y_val = y_train_full.iloc[idx_val]

    # Class-balance sample weights on TRAIN only
    w_train = compute_sample_weight(class_weight="balanced", y=y_train)

    # ==================
    # XGBoost model
    # ==================
    model = xgb.XGBClassifier(
        n_estimators=500, # number of trees (iterations)
        max_depth=3, # how deep each tree can grow
        objective="multi:softprob", # returns class probabiltities for multiclass
        num_class=len(le.classes_), # matches number of labels
        random_state=42,
        eval_metric="mlogloss", # multiclass log loss, used for earlystopping
        early_stopping_rounds=10 # stop if validation doesn't improve for 10 rounds
    )

    # Train
    model.fit(
        X_train,
        y_train,
        sample_weight=w_train, # gives more importance to rare classes
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Fold {fold} accuracy   : {acc:.4f}")
    print(f"Fold {fold} macro F1   : {macro_f1:.4f}")
    print(f"Fold {fold} weighted F1: {weighted_f1:.4f}\n")

    print(classification_report(
        y_test,
        y_pred,
        target_names=le.classes_,
        zero_division=0
    ))

    fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

# =========================================================
# Summary across folds
# =========================================================
results_df = pd.DataFrame(fold_results)

print("\n" + "="*60)
print("CROSS-VALIDATION SUMMARY")
print("="*60)
print(results_df)

print("\nMean metrics:")
print(results_df[["accuracy", "macro_f1", "weighted_f1"]].mean())



FOLD 1
Fold 1 accuracy   : 0.8125
Fold 1 macro F1   : 0.4295
Fold 1 weighted F1: 0.8342

                precision    recall  f1-score   support

Bronchiectasis       0.46      0.67      0.55         9
          COPD       0.95      0.87      0.91       862
       Healthy       0.21      0.67      0.32        33
     Pneumonia       0.10      0.14      0.12        36
          URTI       0.35      0.19      0.25        36

      accuracy                           0.81       976
     macro avg       0.42      0.51      0.43       976
  weighted avg       0.87      0.81      0.83       976


FOLD 2
Fold 2 accuracy   : 0.8732
Fold 2 macro F1   : 0.3911
Fold 2 weighted F1: 0.8759

                precision    recall  f1-score   support

Bronchiectasis       0.30      0.20      0.24        15
          COPD       0.96      0.94      0.95       870
       Healthy       0.43      0.56      0.49        36
     Pneumonia       0.23      0.36      0.28        39
          URTI       0.00      0

## Branch 2: **Simple CNN**

## Concatenate CNN + XGB Features

## Predict